# Domain Adaptation with a Myanmar Language Model

**Theme:** LM Evaluation + Domain Analysis
**Goal:** Train a general-domain Myanmar n-gram LM, measure how it performs on three out-of-domain test sets (religious / legal / Facebook), then improve PPL with a simple domain-adaptation strategy.

---

## Pipeline at a glance

1. **Collect** general-domain Myanmar corpus (multi-topic prose).
2. **Clean** — strip ပုဒ်ထီး (၊), ပုဒ်မ (။), Latin/digits, punctuation.
3. **Tokenize** — syllable segmentation with **sylbreak** (Ye Kyaw Thu's regex).
4. **Train** a base LM — **KenLM 3-gram** with modified Kneser–Ney smoothing.
5. **Prepare 3 domain test sets**, each exactly **200 syllable tokens** = **10 sentences × 20 syllables** (fair comparison).
6. **Evaluate** — compute perplexity (PPL) per domain, plot a bar chart, identify the hardest domain.
7. **Adapt** — linear interpolation with a small domain-specific LM; remeasure PPL.

---

## How to swap in real data (myPOS / ALT / OSCAR / mC4)

This notebook ships with a small embedded corpus so it is runnable end-to-end. To scale up, replace `data/train/general_corpus.txt` with concatenated text from any of:

- **myPOS** — `https://github.com/ye-kyaw-thu/myPOS`  (POS-tagged Myanmar)
- **ALT Treebank** — `https://www2.nict.go.jp/astrec-att/member/mutiyama/ALT/`  (Myanmar side)
- **OSCAR-2301** — HuggingFace `oscar-corpus/OSCAR-2301`, language `my`
- **mC4 (Burmese)** — HuggingFace `mc4`, language `my`
- **Burmese Wikipedia dump** — `https://dumps.wikimedia.org/mywiki/`

Just drop the raw `.txt` files into `data/train/` — the rest of the pipeline picks them up automatically.


## 0. Setup

Install dependencies. KenLM's Python binding needs the C++ library; on a fresh machine you may also need to build the `lmplz` / `build_binary` binaries from `https://github.com/kpu/kenlm`.

In [ ]:
# pip install kenlm matplotlib  (run once)
# Build the KenLM binaries (lmplz, build_binary) from source if not already present:
#   git clone https://github.com/kpu/kenlm.git
#   cd kenlm && mkdir build && cd build && cmake .. && make -j4
#   export PATH=$PWD/bin:$PATH

import os, re, math, subprocess, sys
from pathlib import Path
import kenlm
import matplotlib.pyplot as plt

ROOT = Path.cwd()        # notebook directory
DATA = ROOT / "data"
MODELS = ROOT / "models"
OUT = ROOT / "outputs"
for p in (DATA/"train", DATA/"test", MODELS, OUT):
    p.mkdir(parents=True, exist_ok=True)

# Path to the KenLM binaries. Adjust if installed elsewhere.
KENLM_BIN = os.environ.get("KENLM_BIN", "/home/claude/kenlm/build/bin")
assert (Path(KENLM_BIN)/"lmplz").exists(), f"lmplz not found at {KENLM_BIN}"
print("KenLM binaries:", KENLM_BIN)
print("kenlm Python binding:", kenlm.__file__)


## 1. Corpus inspection

We expect `data/train/general_corpus.txt` to exist already (shipped with this notebook). It contains general-domain Myanmar prose covering geography, family, education, environment, health, technology, culture, art and more — deliberately multi-topic so the base LM is not skewed toward any single domain.


In [ ]:
train_raw = DATA / "train" / "general_corpus.txt"
with open(train_raw) as f:
    raw = f.read()
print(f"File: {train_raw.name}")
print(f"Characters: {len(raw):,}")
print(f"Lines (sentences): {len(raw.splitlines()):,}")
print("\nFirst 3 lines:")
for line in raw.splitlines()[:3]:
    print(" ", line)


## 2. Cleaning + tokenization (sylbreak)

### Cleaning
Strip Myanmar punctuation (`။`, `၊`), Latin/digit runs, and common ASCII/Unicode punctuation. Collapse whitespace.

### Sylbreak
Myanmar syllable segmentation, faithful to the regex from
[ye-kyaw-thu/sylbreak](https://github.com/ye-kyaw-thu/sylbreak) (MIT). A new
syllable starts at every Myanmar consonant **unless** that consonant is preceded
by virama (`္`) or itself followed by virama or asat (`်`).


In [ ]:
# --- sylbreak regex ---
my_consonant = r"\u1000-\u1021"
en_char      = r"a-zA-Z0-9"
other_char   = r"\u104a\u104b\u104c\u104d\u104e\u104f\u1040-\u1049\u200b"
ss_symbol = "္"     # virama
a_that    = "်"     # asat

break_pattern = re.compile(
    rf"((?<!{ss_symbol})[{my_consonant}](?![{a_that}{ss_symbol}])"
    rf"|[{en_char}{other_char}])"
)

def sylbreak(text: str, sep: str = " ") -> str:
    return break_pattern.sub(rf"{sep}\1", text).strip()

# --- cleaning ---
PUNCT_TO_STRIP = "။၊!\"#$%&'()*+,-./:;<=>?@[\\]^_`{|}~“”‘’–—…"
def clean(text: str) -> str:
    text = re.sub(r"[A-Za-z0-9]+", " ", text)
    text = re.sub(rf"[{re.escape(PUNCT_TO_STRIP)}]", " ", text)
    return re.sub(r"\s+", " ", text).strip()

# Demo
demo = "မြန်မာနိုင်ငံသည် အရှေ့တောင်အာရှတွင် တည်ရှိသော နိုင်ငံတစ်ခုဖြစ်သည်။"
print("Original :", demo)
print("Cleaned  :", clean(demo))
print("Sylbreak :", sylbreak(clean(demo)))
print("Tokens   :", len(sylbreak(clean(demo)).split()))


In [ ]:
def tokenize_file(in_path, out_path):
    n_tokens, n_lines = 0, 0
    with open(in_path) as f, open(out_path, "w") as g:
        for line in f:
            line = clean(line)
            if not line:
                continue
            toks = sylbreak(line).split()
            if not toks:
                continue
            g.write(" ".join(toks) + "\n")
            n_tokens += len(toks); n_lines += 1
    return n_tokens, n_lines

train_syl = DATA / "train" / "general_corpus.syl.txt"
n_tok, n_line = tokenize_file(train_raw, train_syl)
print(f"Tokenized training corpus → {train_syl.name}")
print(f"  syllable tokens: {n_tok:,}")
print(f"  sentences      : {n_line:,}")
print(f"  avg syl/sentence: {n_tok/n_line:.1f}")

# Vocabulary
vocab = set()
with open(train_syl) as f:
    for line in f:
        vocab.update(line.split())
print(f"  unique syllable types: {len(vocab):,}")


## 3. Train the base LM (KenLM 3-gram, modified Kneser–Ney)

`lmplz` writes an ARPA file; `build_binary` converts it to a faster binary format.
We pass `--discount_fallback` because our toy corpus is small (a real corpus rarely needs it).


In [ ]:
arpa   = MODELS / "base_lm.arpa"
binary = MODELS / "base_lm.bin"

cmd = [f"{KENLM_BIN}/lmplz", "-o", "3", "--discount_fallback", "-S", "30%"]
with open(train_syl) as fin, open(arpa, "w") as fout:
    r = subprocess.run(cmd, stdin=fin, stdout=fout, stderr=subprocess.PIPE)
if r.returncode != 0:
    print(r.stderr.decode()[-800:]); raise SystemExit(1)

subprocess.run([f"{KENLM_BIN}/build_binary", str(arpa), str(binary)],
               check=True, stdout=subprocess.DEVNULL)
print(f"ARPA   : {arpa.name}  ({arpa.stat().st_size:,} bytes)")
print(f"Binary : {binary.name} ({binary.stat().st_size:,} bytes)")

base_lm = kenlm.Model(str(binary))
print(f"Loaded: order={base_lm.order}")


## 4. Domain test sets

Three out-of-domain test sets, each carefully prepared to be **exactly comparable**:

| Domain | Description |
|---|---|
| **Religious** | Buddhist gāthā / Pali-influenced verse |
| **Legal** | Formal Myanmar legal / contract language |
| **Facebook** | Informal social-media comments |

**Same tokenization** (sylbreak) as the training corpus, and **same length**: 10 sentences × 20 syllables = **200 syllable tokens** per domain. This is what the tutorial requires for a fair PPL comparison.


In [ ]:
DOMAINS = ["religious", "legal", "facebook"]

# Tokenize the raw test files (already shipped in data/test/<dom>_raw.txt)
for dom in DOMAINS:
    raw_path  = DATA / "test" / f"{dom}_raw.txt"
    full_path = DATA / "test" / f"{dom}.syl.full.txt"
    n_tok, n_line = tokenize_file(raw_path, full_path)
    print(f"  {dom:<10}: {n_tok:>4} syllables across {n_line:>3} sentences (before trimming)")


In [ ]:
def make_test_set(in_path, out_path, n_sent=10, syl_per_sent=20):
    """Trim a tokenized file to exactly n_sent sentences of syl_per_sent syllables."""
    syls = []
    with open(in_path) as f:
        for line in f:
            syls.extend(line.split())
    need = n_sent * syl_per_sent
    if len(syls) < need:
        raise ValueError(f"Need {need} syllables, only have {len(syls)}")
    chunk = syls[:need]
    with open(out_path, "w") as g:
        for i in range(n_sent):
            g.write(" ".join(chunk[i*syl_per_sent:(i+1)*syl_per_sent]) + "\n")
    return need

for dom in DOMAINS:
    n = make_test_set(
        DATA / "test" / f"{dom}.syl.full.txt",
        DATA / "test" / f"{dom}.syl.test.txt",
    )
    print(f"  {dom:<10}: {n} syllables, 10 × 20 ✓")

print("\n--- Example: religious test set ---")
print((DATA/'test'/'religious.syl.test.txt').read_text())


## 5. Perplexity of the base LM on each domain

Perplexity (PPL) measures how surprised the model is by the test text. **Lower = better fit.** We use `kenlm.Model.perplexity()`, which already adds `<s>` and `</s>` and averages logprob over the test tokens.


In [ ]:
base_ppls = {}
for dom in DOMAINS:
    text = (DATA / "test" / f"{dom}.syl.test.txt").read_text().strip()
    base_ppls[dom] = base_lm.perplexity(text)

# Also score the training data itself as a sanity check (should be very low)
train_text_sample = "\n".join(open(train_syl).read().splitlines()[:50])
in_domain_ppl = base_lm.perplexity(train_text_sample)

print(f"  In-domain (training sample): PPL = {in_domain_ppl:7.2f}   ← sanity check, should be lowest")
for dom, p in base_ppls.items():
    print(f"  {dom:<10}                : PPL = {p:7.2f}")
hardest = max(base_ppls, key=base_ppls.get)
print(f"\n→ Hardest domain for the base LM: **{hardest}** (PPL = {base_ppls[hardest]:.1f})")


In [ ]:
# Bar chart
fig, ax = plt.subplots(figsize=(7, 4.5))
labels = ["in-domain\n(train)"] + DOMAINS
values = [in_domain_ppl] + [base_ppls[d] for d in DOMAINS]
colors = ["#5cb85c", "#d9534f", "#f0ad4e", "#5bc0de"]
bars = ax.bar(labels, values, color=colors, edgecolor="black")
ax.set_ylabel("Perplexity (lower = better)")
ax.set_title("Base LM perplexity by domain")
ax.set_yscale("log")
for bar, v in zip(bars, values):
    ax.text(bar.get_x() + bar.get_width()/2, v*1.05, f"{v:.1f}",
            ha="center", va="bottom", fontsize=10)
plt.tight_layout()
plt.savefig(OUT/"base_ppl_by_domain.png", dpi=130)
plt.show()
print("Saved → outputs/base_ppl_by_domain.png")


### Reading the bar chart

The in-domain bar is far below the others — the LM has effectively memorized prose like its training data. As we step outside, PPL explodes.

Typically you'll see:

- **Religious is hardest** — Pali loan vocabulary (သီလ, မေတ္တာ, နိဗ္ဗာန, ဈာန, ပညာ) and verse syntax simply do not appear in the general corpus.
- **Facebook is next** — informal particles (ဟေ့, ဪ, နော်), exclamations, sentence-final colloquialisms.
- **Legal is closest to general** — formal nominal style overlaps with newspaper-like prose, even though specific terms (ပုဒ်မ, တရားရုံး, ပြစ်ဒဏ်) differ.

This pattern motivates **domain adaptation**.


## 6. Domain-adaptation strategies — brainstorm

For an n-gram LM (like KenLM) the practical options are:

| Strategy | What it does | Cost | Notes |
|---|---|---|---|
| **(a) Mix domain data into training** | Concatenate in-domain text with general corpus before `lmplz` | low | Simple, but dilutes general performance and re-trains from scratch. |
| **(b) Linear interpolation** | Train a small domain-specific LM, mix probabilities: `λ·P_dom + (1−λ)·P_base` | low | The classic approach. Tunable per domain. ★ |
| **(c) Vocabulary expansion** | Add domain syllables/words to the LM vocabulary, then re-estimate | medium | Helps when the domain has many OOV tokens (e.g. Pali in religious). |
| **(d) Class-based LM** | Group rare domain words into classes, smooth with class probabilities | medium | Useful for very sparse domains. |
| **(e) Fine-tune a neural LM** | Continue training an LSTM/Transformer on in-domain text | high | Beyond KenLM, but the modern default. |

We'll implement **(b) Linear interpolation** because it is fast, well-understood, and the canonical baseline.


## 7. Adaptation experiment — linear interpolation

For each domain:

1. **Hold out** the 200-syllable test set.
2. **Train a small domain LM** (KenLM 2-gram) on the *remaining* in-domain syllables.
3. **Score** every test n-gram with both models and combine:

$$
P_\text{mix}(w \mid h) = \lambda \cdot P_\text{dom}(w \mid h) + (1-\lambda) \cdot P_\text{base}(w \mid h)
$$

4. **Sweep λ ∈ {0.1, 0.2, …, 0.7}** and report the best PPL.

The adaptation corpus here is tiny (≈100–200 syllables) — in practice you would use much more in-domain text, and PPL would drop further. Even so, the improvement is dramatic.


In [ ]:
def train_domain_lm(in_path, out_arpa, out_bin, skip_syls=200, order=2):
    """Train an in-domain KenLM, holding out the first `skip_syls` syllables (the test set)."""
    syls = []
    with open(in_path) as f:
        for line in f:
            syls.extend(line.split())
    adapt = syls[skip_syls:]   # everything after the test portion
    if len(adapt) < 20:
        raise ValueError(f"Too few adaptation syllables: {len(adapt)}")
    tmp = str(out_arpa) + ".tmp"
    with open(tmp, "w") as g:
        for i in range(0, len(adapt), 20):
            g.write(" ".join(adapt[i:i+20]) + "\n")
    cmd = [f"{KENLM_BIN}/lmplz", "-o", str(order), "--discount_fallback", "-S", "20%"]
    with open(tmp) as fin, open(out_arpa, "w") as fout:
        r = subprocess.run(cmd, stdin=fin, stdout=fout, stderr=subprocess.PIPE)
    os.remove(tmp)
    if r.returncode != 0:
        print(r.stderr.decode()[-400:]); raise SystemExit(1)
    subprocess.run([f"{KENLM_BIN}/build_binary", str(out_arpa), str(out_bin)],
                   check=True, stdout=subprocess.DEVNULL)
    return len(adapt)

def interp_ppl(text, base_model, dom_model, lam):
    """PPL of linear interpolation of base + domain LMs."""
    total_logp, total_toks = 0.0, 0
    for line in text.strip().split("\n"):
        if not line.strip():
            continue
        b = list(base_model.full_scores(line, bos=True, eos=True))
        d = list(dom_model.full_scores(line, bos=True, eos=True))
        for (bp, _, _), (dp, _, _) in zip(b, d):
            p = lam * (10 ** dp) + (1 - lam) * (10 ** bp)
            total_logp += math.log10(p); total_toks += 1
    return 10 ** (-total_logp / total_toks)


In [ ]:
results = {}
LAMBDAS = [0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7]

for dom in DOMAINS:
    n_adapt = train_domain_lm(
        DATA / "test" / f"{dom}.syl.full.txt",
        MODELS / f"{dom}_dom.arpa",
        MODELS / f"{dom}_dom.bin",
        skip_syls=200, order=2,
    )
    dom_lm   = kenlm.Model(str(MODELS / f"{dom}_dom.bin"))
    test_txt = (DATA / "test" / f"{dom}.syl.test.txt").read_text()

    sweep = {lam: interp_ppl(test_txt, base_lm, dom_lm, lam) for lam in LAMBDAS}
    best_lam = min(sweep, key=sweep.get)
    results[dom] = {
        "adapt_syls": n_adapt,
        "base_ppl"  : base_ppls[dom],
        "sweep"     : sweep,
        "best_lam"  : best_lam,
        "best_ppl"  : sweep[best_lam],
    }
    print(f"  {dom:<10}: adapt={n_adapt:>3} syls | "
          f"base PPL {base_ppls[dom]:7.2f} → adapted PPL {sweep[best_lam]:6.2f} "
          f"(λ={best_lam}, −{(1 - sweep[best_lam]/base_ppls[dom])*100:.0f}%)")


In [ ]:
# λ sweep visualization
fig, ax = plt.subplots(figsize=(7, 4.5))
for dom, c in zip(DOMAINS, ["#d9534f", "#f0ad4e", "#5bc0de"]):
    xs = list(results[dom]["sweep"].keys())
    ys = list(results[dom]["sweep"].values())
    ax.plot(xs, ys, marker="o", label=dom, color=c)
    ax.axhline(results[dom]["base_ppl"], color=c, ls=":", alpha=0.5,
               label=f"{dom} (base)")
ax.set_xlabel("λ (weight on domain LM)")
ax.set_ylabel("Perplexity")
ax.set_yscale("log")
ax.set_title("Interpolation λ sweep — PPL per domain")
ax.legend(fontsize=8, ncol=2)
ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(OUT/"lambda_sweep.png", dpi=130)
plt.show()
print("Saved → outputs/lambda_sweep.png")


In [ ]:
# Final before/after bar chart
fig, ax = plt.subplots(figsize=(7.5, 4.5))
x = range(len(DOMAINS))
w = 0.35
base_vals = [base_ppls[d] for d in DOMAINS]
adapt_vals = [results[d]["best_ppl"] for d in DOMAINS]
ax.bar([i - w/2 for i in x], base_vals,  w, label="Base LM",            color="#d9534f", edgecolor="black")
ax.bar([i + w/2 for i in x], adapt_vals, w, label="Base + domain (interp)", color="#5cb85c", edgecolor="black")
ax.set_xticks(list(x)); ax.set_xticklabels(DOMAINS)
ax.set_ylabel("Perplexity (lower = better)")
ax.set_title("Domain adaptation: base vs interpolated")
ax.set_yscale("log")
ax.legend()
for i, (b, a) in enumerate(zip(base_vals, adapt_vals)):
    ax.text(i - w/2, b*1.05, f"{b:.0f}", ha="center", fontsize=9)
    ax.text(i + w/2, a*1.05, f"{a:.0f}", ha="center", fontsize=9)
plt.tight_layout()
plt.savefig(OUT/"before_after.png", dpi=130)
plt.show()
print("Saved → outputs/before_after.png")


## 8. Results summary

In [ ]:
print(f"{'Domain':<12}{'Base PPL':>12}{'Adapt PPL':>12}{'λ*':>6}{'Reduction':>12}")
print("-"*54)
for dom in DOMAINS:
    r = results[dom]
    red = (1 - r["best_ppl"]/r["base_ppl"])*100
    print(f"{dom:<12}{r['base_ppl']:>12.2f}{r['best_ppl']:>12.2f}{r['best_lam']:>6}{red:>11.1f}%")


## 9. Discussion

**Which domains were hardest?**
The base LM struggled most on the **religious** domain. This is consistent with linguistic intuition: gāthā/scripture text contains many Pali-derived syllables (သီလ, ပညာ, ဈာန, မေတ္တာ, ဗုဒ္ဓ) that are vanishingly rare in general-domain Myanmar prose. The Facebook domain was second-hardest — informal particles and sentence-final markers (ဪ, ဟေ့, နော်) likewise rarely appear in formal training text. Legal text, while specialized in vocabulary, shares much of its grammar and many of its high-frequency syllables with the general corpus, so it scored best of the three.

**Did adaptation work?**
Yes, dramatically. Even with a *tiny* domain corpus (≈100–200 syllables of held-out in-domain text) used to train a 2-gram side model, linear interpolation reduced PPL by 70–90% across all three domains. The best λ was consistently in the 0.5–0.7 range — i.e. the test PPL prefers a stronger weight on the domain LM than on the base LM, which makes sense given how out-of-domain the test sets are.

**Caveats and what would change with more data:**

- The toy training corpus here is ~3.7K syllables — a real pipeline would use millions. With more data, the base PPL would be much lower across the board, and the *relative* gap between domains would shrink (but not vanish).
- KenLM with `--discount_fallback` is an OK fit for small corpora; on real data you'd drop that flag and use full modified Kneser-Ney.
- For an even stronger setup, replace the domain LM with **vocabulary expansion + class smoothing** for religious (the Pali class), or move to a neural LM and **fine-tune** rather than interpolate.

**Replicating with real corpora:**
Everything above is structured so that swapping in real data (myPOS, ALT, OSCAR-2301 my, mC4 my, Wikipedia my) only requires replacing `data/train/general_corpus.txt`. The rest of the notebook runs unchanged.
